# MQT Fidelity Project — 10k Corpus Driver (non-amp-amp headline run)

Generates ~10,100 circuits across 15 algorithms (excluding grover/qwalk),
computes Hellinger fidelity under FakeBrisbane at 8192 shots, extracts pre- and
post-transpile features, and runs the full supervised harness.

## Runtime / setup requirements

**Use a CPU runtime, not GPU**: Qiskit Aer's noisy simulation at N=3-12 is CPU-bound
(statevector + noise channels). GPU acceleration would lose to memory-transfer overhead
for ~10k small circuits. Set `Runtime → Change runtime type → CPU` (High-RAM CPU if available).

**Colab Pro 24-hour session**: This run is ~17 hours wallclock. Pro's 24-hour session is
necessary; Free's 12-hour kill would interrupt. Keep the browser tab open and laptop awake.
Pro+ would offer background execution, but Pro plus an overnight run works.

## Excluded by design

**grover and qwalk** — amp-amp algorithms whose pre-transpile depth scales exponentially
under multi-controlled-gate decomposition on heavy-hex topology, producing at-floor
fidelities for N≥10. Reported separately as the OOD/structural-failure-mode chapter.

## Key features

- **Local-SSD scratch + periodic Drive rsync**: All scripts read/write the fast local
  `/content/MQT/data/` SSD; a background thread rsyncs to Drive every 10 minutes.
  Avoids FUSE-to-Drive latency on small-file writes (~50% speedup vs symlink-to-Drive).
- **MQT Bench seed-leak bypass**: `loader.py` uses `random_parameters=False` then assigns
  symbolic parameters with `np.random.default_rng(seed)`, bypassing the hardcoded
  `default_rng(10)` at `benchmark_generation.py:81`. Validator in Cell 3 confirms.
- **Resumability**: Loader, fidelity, and post-transpile feature extraction all skip
  already-completed work on restart. If the session disconnects, restart from where you left.

## Cell 1: Setup — clone repo, mount Drive, pull any existing state

Code lives on local SSD (`/content/MQT/`). Data starts on local SSD too, with previous
session state pulled from Drive at the start, and a background thread syncing local→Drive
every 10 minutes. This avoids FUSE-on-every-small-write.

In [ ]:
# Mount Drive (persistence target)
from google.colab import drive
drive.mount('/content/drive')

# Clone code to local SSD (fast)
!rm -rf /content/MQT
!git clone -b cleanup/pre-10k-tidy https://github.com/StrelzoffUmich/SIADS696-mqtbench-circuitfidelity.git /content/MQT

import os
os.chdir('/content/MQT')
!pwd

# Pull any existing run state from Drive (resumes from previous session)
!mkdir -p /content/drive/MyDrive/MQT_data
!mkdir -p /content/MQT/data
!rsync -a --info=stats2 /content/drive/MyDrive/MQT_data/ /content/MQT/data/
!ls -la /content/MQT/data/ 2>/dev/null

In [ ]:
# Install dependencies. Should take 2-3 minutes on first run.
!pip install -q -r requirements.txt
!python -c "import mqt.bench; import qiskit_aer; from qiskit_ibm_runtime.fake_provider import FakeBrisbane; print('OK')"

In [ ]:
# Start the background sync thread: rsync local→Drive every 10 minutes.
# Lose at most 10 minutes of in-flight work on a disconnect.
import subprocess, time, threading
_sync_stop = threading.Event()

def _sync_loop():
    while not _sync_stop.is_set():
        try:
            subprocess.run(
                ['rsync', '-a', '--delete-after',
                 '/content/MQT/data/', '/content/drive/MyDrive/MQT_data/'],
                check=False, capture_output=True, timeout=300,
            )
        except Exception:
            pass
        _sync_stop.wait(timeout=600)

_sync_thread = threading.Thread(target=_sync_loop, daemon=True)
_sync_thread.start()
print('Background sync started (every 10 min, local→Drive).')

## Cell 2: Session-length probe (optional but recommended)

Colab Pro's session limit has been documented ambiguously. Before committing 17 hours,
verify with a short timestamp loop that the session is actually long-running and not
subject to idle-detection. Sleeps for 3 minutes printing every 15s — if Colab is going
to kill the kernel for inactivity, this will surface it cheaply.

Skip this cell if you trust the Pro 24-hour claim.

In [ ]:
import time
from datetime import datetime
t0 = time.time()
for _ in range(12):
    print(f'{datetime.now().isoformat()}  elapsed={time.time()-t0:.0f}s', flush=True)
    time.sleep(15)
print('OK — session alive after 3 minutes of zero browser interaction.')

## Cell 3: Verify MQT Bench seed-leak bypass

Sixteen-check regression test. Confirms the patched `loader.py` actually produces parameter
variation across seeds — without this, the entire 10k corpus would silently collapse to ~100
unique circuits.

**Expected output**: all checks PASS, exit 0. If FAIL, do not proceed — the MQT Bench
upstream code has changed and the workaround needs revisiting.

In [ ]:
!python src/load/validate_seed_propagation.py

## Cell 4: Generate corpus

27 algorithms × N=3..12, seed variation as appropriate per algorithm:

- **7 parameterizable** (qaoa, vqe_*, qnn, bmw_quark_*): 200 seeds each via patched loader
- **19 deterministic** (ghz, qft, hhl, arithmetic family, etc.): seeds collapse to 1 per N via dedup; restricted-N algos auto-skipped at unsupported N
- **graphstate**: ~3 unique per N via internal randomness

Expected total: ~13,000-14,000 unique QASM files. Wall time: ~10-15 minutes.

**Excluded**:
- grover, qwalk (amp-amp; computationally infeasible on Aer due to multi-controlled-gate decomposition blow-up — separate OOD chapter)
- seven_qubit_steane_code, shor, shors_nine_qubit_code (require N divisible by 13 / specific qiskit-defined values / 17; none feasible in our N=3..12 range)
- ae, ghz_dynamic (mqt.bench  succeeds but  fails — ae has numpy array conversion issue, ghz_dynamic uses dynamic-circuit conditionals unsupported by OpenQASM 2)

In [ ]:
ALGORITHMS = [
    # Parameterizable, all N=3..12 (6) — produce K unique circuits per N via patched loader
    'qaoa', 'vqe_su2', 'vqe_real_amp', 'vqe_two_local', 'qnn',
    'bmw_quark_cardinality',
    # Parameterizable, even N only (1) — bmw_quark_copula
    'bmw_quark_copula',
    # Truly deterministic, all N=3..12 (10) — collapse to 1 per N via dedup
    'bv', 'dj', 'ghz', 'hhl',
    'qft', 'qftentangled', 'qpeexact', 'qpeinexact', 'randomcircuit', 'wstate',
    # Truly deterministic, restricted N (loader skips unsupported N automatically) (9)
    'full_adder', 'cdkm_ripple_carry_adder', 'draper_qft_adder', 'modular_adder',
    'half_adder', 'hrs_cumulative_multiplier',
    'multiplier', 'rg_qft_multiplier', 'vbe_ripple_carry_adder',
    # Internal randomness — varies with global np.random state (1)
    'graphstate',
]
algo_arg = ' '.join(ALGORITHMS)

!python src/load/loader.py \
    --algorithms {algo_arg} \
    --qubits 3 12 \
    --n-seeds 200 \
    --out data/qasm

!ls data/qasm | wc -l
!ls data/qasm | head -10

## Cell 5: Compute fidelity labels

Hellinger fidelity under FakeBrisbane noise model, **8192 shots** (8× the pilot's 1024 —
drives per-circuit fidelity SE from ~0.0075 down to ~0.0025, improving signal-to-noise
by ~3×).

Per-circuit timeout: 600s. Resumable: writes `data/fidelity.csv` row-by-row; existing
rows preserved on restart.

Wall time: **~15-20 hours on 4-core Colab CPU**. The bulk of the run.

In [ ]:
!python src/load/fidelity.py \
    --save-counts \
    --shots 8192 \
    --timeout 600 \
    --max-qubits 12 \
    --out data/fidelity.csv

import pandas as pd
df = pd.read_csv('data/fidelity.csv')
print(f'Total rows: {len(df)}')
print(f'Successful: {df["fidelity"].notna().sum()}')
print(f'Failed (timeout/error): {df["fidelity"].isna().sum()}')
print(df.groupby('algo')['fidelity'].agg(['count', 'mean', 'std']).round(3))

## Cell 6: Extract pre-transpile features

Runs `features.extract()` on the abstract circuit (algorithmic intent layer).
Fast — ~1 minute on the full 10k. Output: `data/features.csv`, 38 feature columns + metadata.

In [ ]:
!python src/feature_sel/extract_features.py

## Cell 7: Extract post-transpile features

Transpiles each circuit to FakeBrisbane (heavy-hex 127-qubit, basis ECR/RZ/SX/X),
compresses to active-qubit subgraph (avoids 127-qubit-graph artifacts), then extracts
features on the compiler-realized circuit.

Resumable. Wall time: ~30 min for the 10k corpus (no amp-amp circuits in scope, no slow tail).

Output: `data/features_post.csv`, same schema as `features.csv`.

In [ ]:
!python src/feature_sel/extract_post_transpile.py

## Cell 8: Apply combined at-floor classification

Refits the `at_floor` column using the combined rule: chi-square p ≥ 0.05 **AND**
fidelity < 0.10. The chi-square-alone rule mislabels algorithms with naturally-uniform
ideal distributions (graphstate) as at-floor; the combined rule corrects this.

Fast — <1 minute.

In [ ]:
!python src/load/refit_at_floor.py

## Cell 9: Run supervised harness

80 cells of GridSearchCV-tuned R²: 5 feature views × 4 models × 2 CVs × 2 strata,
with bootstrap-over-folds 95% CIs. Plus Stage A vote-across-models decision rule,
Stage B slope test, Stage C signal-vs-noise-floor check, per-algo attribution.

Wall time: **~45-90 minutes** on the 10k corpus (vs ~30 min on the pilot — scales with
K-fold model fitting cost).

In [ ]:
!python src/feature_sel/experiment_full_pilot.py 2>&1 | tee data/results/full_pilot.log

## Cell 10: Diagnostic v2 + figures

Runs the diagnostic re-pass (bounded-RF per-algo attribution + median slope test for
robustness), then generates the five pilot figures + two v2-corrected figures.

Wall time: ~5 minutes.

In [ ]:
!python src/feature_sel/diagnose_pilot.py
!python src/feature_sel/visualize_full_pilot.py
!python src/feature_sel/visualize_pilot_v2.py
!ls data/results/figures/

## Cell 11: Final sync + bundle for download

Stops the background sync thread, does one final manual rsync to Drive (so the latest
results are persisted), then packages CSVs + figures + verdict files into a single zip
for direct download via Colab's file panel.

In [ ]:
# Stop the background sync thread and do one final manual rsync
_sync_stop.set()
_sync_thread.join(timeout=10)
!rsync -a --info=stats2 /content/MQT/data/ /content/drive/MyDrive/MQT_data/

# Bundle for download
import shutil
from pathlib import Path

bundle = Path('data/results_10k')
if bundle.exists():
    shutil.rmtree(bundle)
bundle.mkdir()

for src in ['data/features.csv', 'data/features_post.csv', 'data/fidelity.csv', 'data/results']:
    s = Path(src)
    if s.is_dir():
        shutil.copytree(s, bundle / s.name)
    elif s.is_file():
        shutil.copy(s, bundle / s.name)

shutil.make_archive(str(bundle), 'zip', bundle)
print(f'Bundle: data/results_10k.zip '
      f'({Path("data/results_10k.zip").stat().st_size / 1e6:.1f} MB)')
print('Drive copy at: /content/drive/MyDrive/MQT_data/')

## After this run completes

Next: run the amp-amp OOD chapter. Clone this notebook, change `ALGORITHMS` in Cell 4 to
`['grover', 'qwalk']`, point `--out` to a separate path (`data_amp_amp/qasm`), and adjust
the `MQT_data` Drive directory accordingly.

**Timing probe before committing the OOD run**: at K=200 the amp-amp run is ~4000 circuits
but per-circuit cost dominates. Run a quick `--n-seeds 1` sanity check at N=10-12 first
(should produce ~20-30 circuits, time them) — multiply by 200 to estimate the full sweep
cost. Likely 3-5 hours wall time but worth verifying.